In [1]:
from langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model="gpt-5-mini")

## Tools

In [2]:
from langchain.tools import tool

In [3]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about current events or general knowledge. """

    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)

    return response


In [4]:
tool_duckduckgo_search.invoke("What is the capital of France?")

C:\Users\Lavanya Rajesh\AppData\Local\Temp\ipykernel_15992\2405146460.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


'The Île-de-France (/ ˌiːl də ˈfrɒ̃s /; French: [il də fʁɑ̃s] ⓘ; lit. \'Island of France\') is the most populous of the eighteen regions of France, with an official estimated population of 12,271,794 residents on 1 January 2023. [1] Containing the capital city of France, Paris, it is located in the north-central part of the country and often called the Paris Region[3] (French ... Learn why Paris is the capital of France and how it became the political and cultural hub of the country. Discover the best places to visit, the weather, the food, and the nicknames of the "City of Light" and the "City of Love". France, a country of northwestern Europe, is historically and culturally among the most important countries in the Western world. It has also played a highly significant role in international affairs for centuries. Its capital is Paris, one of the most important cultural and commercial centers in the world. The capital of France is Paris, which is the political, economic, cultural, and

In [7]:
@tool 
def tool_wikipedia_search(query: str) -> str:
    """Use this tool when you need to answer questions about general knowledge."""

    import wikipediaapi

    wiki = wikipediaapi.Wikipedia(
        language='en',
        user_agent='RajeshAIProject/1.0 (rajeshetl78456@xyz.com)'
    )

    page = wiki.page(query)

    if not page.exists():
        return f"No Wikipedia page found for: {query}"

    return page.summary


In [8]:
tool_wikipedia_search.invoke("Sachin Tendulkar")

'Sachin Ramesh Tendulkar ( ; Marathi: [sətɕin t̪eɳɖulkəɾ]; born 24 April 1973) is an Indian former international cricketer who captained the Indian national team. Tendulkar is widely regarded as one of the greatest cricketers in the history of cricket. He holds several world records, including being the all-time highest run-scorer in international cricket, receiving the most player of the match awards in international cricket, and being the only batsman to score 100 international centuries. Tendulkar was a Member of Parliament, Rajya Sabha by presidential nomination from 2012 to 2018.\nTendulkar took up cricket at the age of eleven, made his Test match debut on 15 November 1989 against Pakistan in Karachi at the age of sixteen, and went on to represent Mumbai domestically and India internationally for over 24 years. In 2002, halfway through his career, Wisden ranked him the second-greatest Test batsman of all time, behind Don Bradman, and the second-greatest ODI batsman of all time, be

In [10]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "Rajesh Venkatesan",
        "age": 34,
        "occupation": "Data Scientist"
    },
    {
        "name": "Lavanya Pugazhendi",
        "age": 31,
        "occupation": "Cyber Security Analyst"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."


In [11]:
tool_personal_info.invoke("Lavanya Pugazhendi")

'Lavanya Pugazhendi is 31 years old and works as a Cyber Security Analyst.'

In [12]:
@tool
def tool_rag(query: str) -> str:
    """Use this tool when you need to answer questions based on NovaSpehere Organization's documentation.""" 

    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings
    embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
    chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)

    # Retrieve relevant documents from the vector store
    relevant_docs = chroma_db_con.similarity_search(query, k=2)
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])
    return relevant_docs_content
    

In [13]:
tool_rag.invoke("tell me about the france")

C:\Users\Lavanya Rajesh\AppData\Local\Temp\ipykernel_15992\3480324300.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)


'are linked by the Channel Tunnel,\nlocated underneath the English Channel . 88\nFRANCE\nFRANCE\n89\nParis  has  a lot  to  show  \nthe tourists  that  come  by: \nThe  Eiffel  Tower,  Louvre, \nand  the  Palace  of  Versailles.\nare linked by the Channel Tunnel,\nlocated underneath the English Channel . 88\nFRANCE\n89\nParis  has  a lot  to  show  \nthe tourists  that  come  by: \nThe  Eiffel  Tower,  Louvre, \nand  the  Palace  of  Versailles.'

## Bind Tools

In [14]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_personal_info,
    tool_rag
]

In [15]:
llm_bind = llm.bind_tools(toolkit)

In [16]:
llm_bind.invoke("What's the age of Lavanya Pugazhendi?. Make tool calls if necessary.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 284, 'total_tokens': 378, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EP5pVP0YmvstInkXD3oR8CAl5Ovyw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0af69-6e31-7eb1-8025-970ed1c72d57-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'Lavanya Pugazhendi'}, 'id': 'call_wcljHo1fGxC1hfeTTYkNT5WK', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 284, 'output_tokens': 94, 'total_tokens': 378, 'input_token_detail

## ReAct Agent

In [17]:
from langchain.agents import create_agent


my_agent = create_agent(llm_bind, toolkit)

In [18]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the age of John Doe?. Make tool calls if necessary."}]}
)

{'messages': [HumanMessage(content="What's the age of John Doe?. Make tool calls if necessary.", additional_kwargs={}, response_metadata={}, id='386da841-bdb2-41fc-aec6-84980fe637c1'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 280, 'total_tokens': 306, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EP5pcXbft89WK61bvNc9oA0w78t8U', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0af69-8927-78f3-8cb5-8e49667a7077-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': '

In [19]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the age of Lavanya Pugazhendi?"}]}
)

{'messages': [HumanMessage(content="What's the age of Lavanya Pugazhendi?", additional_kwargs={}, response_metadata={}, id='e9d97c5e-7588-41b1-9749-fd6951e56f97'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 278, 'total_tokens': 308, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EP5po8kpc1MH4QpNoIDJFqW9whIBU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0af69-b6da-7360-9331-fd9a682ca017-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'Lavanya Pugazhendi'}, 'id': 'call_kUBLiG